# 📊 Dashboard EMICRON 2023: Análisis de Micronegocios

Este reporte ejecutivo presenta tres dimensiones clave de los micronegocios colombianos en 2023: formalización empresarial, acceso al crédito y crecimiento de ingresos. Las estimaciones se ponderan con el factor oficial de expansión `fex_c` y se calculan a partir de `data/processed/emicron_core.parquet`.

- **Formalización:** promedio ponderado de cuatro prácticas: registro en el RUT, Cámara de Comercio, contabilidad formal y declaración de impuestos.
- **Crédito:** tasa de aprobación entre los negocios que solicitaron crédito, comparada por sector y tamaño.
- **Crecimiento:** variación entre los ingresos de 2023 anualizados y los ingresos de 2022, agregados con `fex_c`.

Los nombres de departamentos y sectores se presentan de forma legible. En el gráfico de ciudades se conserva el código de área reportado en el microdato.

In [1]:
from pathlib import Path

from IPython.display import HTML, display
import pandas as pd
import plotly.express as px

CIUDAD_NOMBRES = {
    "05": "Medellín AM", "08": "Barranquilla AM", "11": "Bogotá",
    "13": "Cartagena", "15": "Tunja", "17": "Manizales AM",
    "18": "Florencia", "19": "Popayán", "20": "Valledupar",
    "23": "Montería", "27": "Quibdó", "41": "Neiva",
    "44": "Riohacha", "47": "Santa Marta", "50": "Villavicencio",
    "52": "Pasto", "54": "Cúcuta AM", "63": "Armenia",
    "66": "Pereira AM", "68": "Bucaramanga AM", "70": "Sincelejo",
    "73": "Ibagué", "76": "Cali AM", "88": "San Andrés",
}

DEPARTAMENTO_NOMBRES = {
    "05": "Antioquia", "08": "Atlántico", "11": "Bogotá", "13": "Bolívar",
    "15": "Boyacá", "17": "Caldas", "18": "Caquetá", "19": "Cauca",
    "20": "Cesar", "23": "Córdoba", "25": "Cundinamarca", "27": "Chocó",
    "41": "Huila", "44": "La Guajira", "47": "Magdalena", "50": "Meta",
    "52": "Nariño", "54": "Norte de Santander", "63": "Quindío",
    "66": "Risaralda", "68": "Santander", "70": "Sucre", "73": "Tolima",
    "76": "Valle del Cauca", "88": "San Andrés",
}

SECTOR_NOMBRES = {
    "01": "Agricultura", "02": "Minería", "03": "Industria manufacturera",
    "04": "Construcción", "05": "Comercio", "06": "Transporte",
    "07": "Alojamiento y comida", "08": "Información y comunicaciones",
    "09": "Inmobiliarias y profesionales", "10": "Educación", "11": "Salud",
    "12": "Artes y entretenimiento",
}

parquet_name = Path("data/processed/emicron_core.parquet")
project_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / parquet_name).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "No se encontró data/processed/emicron_core.parquet desde el directorio actual."
    )

parquet_path = project_root / parquet_name
core = pd.read_parquet(parquet_path)

required_core_columns = {
    "directorio", "secuencia_p", "secuencia_encuesta", "fex_c", "area",
    "cod_depto", "grupos12", "p1633", "p1055", "p640", "p2991",
    "p2992", "p2993", "p1765", "p1568", "p3032_1",
}
missing_core_columns = sorted(required_core_columns.difference(core.columns))
if missing_core_columns:
    raise ValueError(
        "El Parquet no contiene columnas necesarias para los gráficos 1 y 2: "
        + ", ".join(missing_core_columns)
    )

core["fex_c"] = pd.to_numeric(core["fex_c"], errors="coerce")
core = core.loc[core["fex_c"].gt(0)].copy()


## 1. Ciudades con Mayor Formalización Empresarial

San Andrés e Ibagué encabezan el ranking. El índice resume, con ponderación poblacional, la adopción de prácticas de formalización; las áreas se muestran con el código disponible en el microdato.

In [2]:
formal = core[[
    "area", "fex_c", "p1633", "p1055", "p640", "p2991", "p2992", "p2993"
]].copy()
formal["area"] = formal["area"].astype("string").str.strip()
formal = formal.loc[formal["area"].notna() & formal["area"].ne("")].copy()

formal_components = {
    "rut": pd.to_numeric(formal["p1633"], errors="coerce").eq(1),
    "camara": pd.to_numeric(formal["p1055"], errors="coerce").eq(1),
    "contabilidad_formal": pd.to_numeric(formal["p640"], errors="coerce").isin([1, 2, 4]),
    "declaracion_impuestos": pd.concat(
        [pd.to_numeric(formal[column], errors="coerce").eq(1)
         for column in ["p2991", "p2992", "p2993"]],
        axis=1,
    ).any(axis=1),
}
for column, values in formal_components.items():
    formal[column] = values.fillna(False).astype("int8")

formal["indice_formalizacion_pct"] = formal[
    list(formal_components)
].mean(axis=1) * 100
formal["indice_ponderado"] = formal["indice_formalizacion_pct"] * formal["fex_c"]

formalizacion_ciudad = (
    formal.groupby("area", as_index=False)
    .agg(puntaje_ponderado=("indice_ponderado", "sum"), peso_total=("fex_c", "sum"))
)
formalizacion_ciudad["indice_formalizacion_pct"] = (
    formalizacion_ciudad["puntaje_ponderado"] / formalizacion_ciudad["peso_total"]
)
formalizacion_top10 = formalizacion_ciudad.nlargest(10, "indice_formalizacion_pct")
formalizacion_top10 = formalizacion_top10.sort_values(
    "indice_formalizacion_pct", ascending=False
)
formalizacion_top10["ciudad"] = (
    formalizacion_top10["area"].map(CIUDAD_NOMBRES)
    .fillna(formalizacion_top10["area"])
)


In [3]:
fig_formalizacion = px.bar(
    formalizacion_top10,
    x="indice_formalizacion_pct",
    y="ciudad",
    orientation="h",
    text="indice_formalizacion_pct",
    title="Top 10 ciudades por índice de formalización",
    labels={
        "ciudad": "Ciudad",
        "indice_formalizacion_pct": "Índice de formalización (%)",
    },
    color="indice_formalizacion_pct",
    color_continuous_scale=px.colors.sequential.Teal,
    template="plotly_white",
)
fig_formalizacion.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
)
fig_formalizacion.update_layout(
    title={"x": 0, "xanchor": "left", "font": {"size": 20}},
    font={"family": "Arial, sans-serif", "size": 14, "color": "#243042"},
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF",
    xaxis={"title": "Índice de formalización (%)", "range": [0, 100], "showgrid": True, "gridcolor": "#E8EDF2"},
    yaxis={"title": "Ciudad", "categoryorder": "array", "categoryarray": formalizacion_top10["ciudad"].tolist(), "autorange": "reversed"},
    coloraxis_showscale=False,
    margin={"l": 145, "r": 55, "t": 75, "b": 55},
)
display(HTML(fig_formalizacion.to_html(full_html=False, include_plotlyjs="cdn")))

## 2. Brecha de Aprobación de Crédito por Sector

La comparación muestra la proporción ponderada de solicitudes aprobadas entre micronegocios que solicitaron crédito. Las barras separan los negocios unipersonales de aquellos con empleados para destacar diferencias entre sectores y tamaños.

In [4]:
credit = core[["grupos12", "p3032_1", "p1765", "p1568", "fex_c"]].copy()
credit["grupos12"] = credit["grupos12"].astype("string").str.strip()
credit["sector"] = credit["grupos12"].map(SECTOR_NOMBRES).fillna(credit["grupos12"])
credit["p3032_1"] = pd.to_numeric(credit["p3032_1"], errors="coerce")
credit["p1765"] = pd.to_numeric(credit["p1765"], errors="coerce")
credit["p1568"] = pd.to_numeric(credit["p1568"], errors="coerce")
credit = credit.loc[
    credit["sector"].notna()
    & credit["p1765"].eq(1)
    & credit["fex_c"].gt(0)
].copy()
credit["tamano"] = credit["p3032_1"].fillna(0).gt(0).map(
    {True: "Con empleados", False: "Unipersonal"}
)
credit["peso_aprobado"] = credit["fex_c"].where(credit["p1568"].eq(1), 0)

credito_sector = (
    credit.groupby(["grupos12", "sector", "tamano"], as_index=False)
    .agg(aprobados_ponderados=("peso_aprobado", "sum"), solicitantes_ponderados=("fex_c", "sum"))
)
credito_sector["tasa_aprobacion_pct"] = (
    100 * credito_sector["aprobados_ponderados"]
    / credito_sector["solicitantes_ponderados"]
)
credito_sector = credito_sector.sort_values("grupos12")


In [5]:
fig_credito = px.bar(
    credito_sector,
    x="sector",
    y="tasa_aprobacion_pct",
    color="tamano",
    barmode="group",
    text="tasa_aprobacion_pct",
    title="Tasa de aprobación de crédito por sector y tamaño del negocio",
    labels={
        "sector": "Sector económico",
        "tasa_aprobacion_pct": "Solicitudes aprobadas (%)",
        "tamano": "Tipo de negocio",
    },
    category_orders={
        "sector": list(SECTOR_NOMBRES.values()),
        "tamano": ["Unipersonal", "Con empleados"],
    },
    color_discrete_sequence=px.colors.qualitative.Bold,
    template="plotly_white",
)
fig_credito.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
)
fig_credito.update_layout(
    title={"x": 0, "xanchor": "left", "font": {"size": 20}},
    font={"family": "Arial, sans-serif", "size": 14, "color": "#243042"},
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF",
    xaxis={"title": "Sector económico", "tickangle": -25, "showgrid": False},
    yaxis={"title": "Solicitudes aprobadas (%)", "range": [0, 110], "ticksuffix": "%", "showgrid": True, "gridcolor": "#E8EDF2"},
    legend={"title": "Tipo de negocio", "orientation": "h", "y": 1.12, "x": 1, "xanchor": "right"},
    margin={"l": 70, "r": 35, "t": 115, "b": 125},
)
display(HTML(fig_credito.to_html(full_html=False, include_plotlyjs="cdn")))

## 3. Top 10 Departamentos con Mayor Crecimiento Interanual

El ranking compara los ingresos mensuales de 2023 anualizados (`ventas_mes_anterior × 12`) con los ingresos de 2022. La variación se calcula sobre totales ponderados por `fex_c`, lo que permite contrastar el crecimiento estimado entre departamentos.

In [6]:
required_growth_columns = {
    "cod_depto", "fex_c", "ventas_mes_anterior", "ventas_anio_anterior"
}
missing_growth_columns = sorted(required_growth_columns.difference(core.columns))
if missing_growth_columns:
    raise ValueError(
        "El Parquet no contiene columnas necesarias para el crecimiento interanual: "
        + ", ".join(missing_growth_columns)
    )

growth = core[
    ["cod_depto", "fex_c", "ventas_mes_anterior", "ventas_anio_anterior"]
].copy()
growth["cod_depto"] = growth["cod_depto"].astype("string").str.strip()
for column in ["ventas_mes_anterior", "ventas_anio_anterior"]:
    growth[column] = pd.to_numeric(growth[column], errors="coerce")

growth = growth.loc[
    growth["cod_depto"].notna()
    & growth["cod_depto"].ne("")
    & growth["fex_c"].gt(0)
    & growth["ventas_mes_anterior"].notna()
    & growth["ventas_anio_anterior"].gt(0)
].copy()
growth["ingresos_2023_anualizados_ponderados"] = (
    growth["ventas_mes_anterior"] * 12 * growth["fex_c"]
)
growth["ingresos_2022_ponderados"] = (
    growth["ventas_anio_anterior"] * growth["fex_c"]
)

growth_by_department = (
    growth.groupby("cod_depto", as_index=False)
    .agg(
        ingresos_2023_ponderados=("ingresos_2023_anualizados_ponderados", "sum"),
        ingresos_2022_ponderados=("ingresos_2022_ponderados", "sum"),
        muestra=("fex_c", "size"),
    )
)
growth_by_department["variacion_interanual_pct"] = (
    100
    * (growth_by_department["ingresos_2023_ponderados"]
       - growth_by_department["ingresos_2022_ponderados"])
    / growth_by_department["ingresos_2022_ponderados"]
)
top10_crecimiento = growth_by_department.nlargest(
    10, "variacion_interanual_pct"
).sort_values("variacion_interanual_pct", ascending=False)
top10_crecimiento["departamento"] = (
    top10_crecimiento["cod_depto"].map(DEPARTAMENTO_NOMBRES)
    .fillna(top10_crecimiento["cod_depto"])
)


In [7]:
fig_crecimiento = px.bar(
    top10_crecimiento,
    x="variacion_interanual_pct",
    y="departamento",
    orientation="h",
    text="variacion_interanual_pct",
    title="Departamentos con mayor crecimiento interanual de ingresos",
    labels={
        "departamento": "Departamento",
        "cod_depto": "Código de departamento",
        "variacion_interanual_pct": "Crecimiento interanual (%)",
        "ingresos_2023_ponderados": "Ingresos 2023 anualizados y ponderados",
        "ingresos_2022_ponderados": "Ingresos 2022 ponderados",
        "muestra": "Registros de la muestra",
    },
    color="variacion_interanual_pct",
    color_continuous_scale=px.colors.sequential.Teal,
    template="plotly_white",
    hover_data={
        "cod_depto": True,
        "ingresos_2023_ponderados": ":,.0f",
        "ingresos_2022_ponderados": ":,.0f",
        "muestra": True,
    },
)
fig_crecimiento.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
)
fig_crecimiento.update_layout(
    title={"x": 0, "xanchor": "left", "font": {"size": 20}},
    font={"family": "Arial, sans-serif", "size": 14, "color": "#243042"},
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF",
    xaxis={"title": "Crecimiento interanual (%)", "rangemode": "tozero", "showgrid": True, "gridcolor": "#E8EDF2"},
    yaxis={"title": "Departamento", "categoryorder": "array", "categoryarray": top10_crecimiento["departamento"].tolist(), "autorange": "reversed"},
    coloraxis_showscale=False,
    margin={"l": 145, "r": 55, "t": 75, "b": 55},
)
display(HTML(fig_crecimiento.to_html(full_html=False, include_plotlyjs="cdn")))